# Hi! 👋

The following notebook is used to **analyze multiple experiments** at once. If you are running the following notebook in Colab, the next cell will ask you for a GDrive URL and a github token. The token will be used to clone [DecentralizedLearning/dEngine](https://github.com/DecentralizedLearning/dEngine). If you don't own the repo, the token can be generated using Github web (`settings > Developer Settings > Personal Access Tokens`). The GDrive URL will be provided by who ran the experiments. 

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import gdown

try:  # Check if we are in Google colab
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
    
    !git clone --quiet https://github.com/DecentralizedLearning/dEngine ..
    !pip install --quiet git+https://github.com/DecentralizedLearning/notebooks.git
    
    EXPERIMENT_GDRIVE_URL = input('Experiment remote URL \t')
    LOGS_PATH = Path('/content/logs')
    if not os.path.exists(LOGS_PATH):
        gdown.download_folder(url=EXPERIMENT_GDRIVE_URL, output=str(LOGS_PATH))
        !unzip -q {LOGS_PATH}/\*.zip -d {LOGS_PATH}    
except:
    pass

<br><br>
<hr>
<br><br>

# Experiment Selection

The next cell will open a file selection widget.

Please select the directory containing the experiment logs (usually inside a `log/` folder).

**Note**: If you are using Google Drive, you may need to navigate one level up. Logs are typically downloaded to `/content/logs`, while the code runs in `/content/SaiSim`.

In [ ]:
from dnotebooks.widgets import StyledExperimentSelectionWidget

experiment_selection_widget, get_selected_experiments = StyledExperimentSelectionWidget()
display(experiment_selection_widget)

<br>
The next cell will open a dropdown selection widget.

Please select a confusion matrix file. These files can be found in `<selected_experiment>/metrics/<node_id>/` and they contain NumPy arrays generated by the simulation callbacks.

You can also load them manually with NumPy:
```
import numpy as np
from saisim.config.constants import METRICS_DIR_NAME

experiment_root, linestyle = get_selected_experiments()[0]
np.load((experiment_root / METRICS_DIR_NAME) / "<node_id>/<filename>.npy")
```

In [ ]:
from dnotebooks.widgets import ConfusionMatrixPartitionMultiSelection

selected_experiments, line_styles = zip(*get_selected_experiments())
line_styles = {exp.name: s for exp, s in zip(selected_experiments, line_styles)}
confusion_matrix_partition_selection_wg, get_confusion_matrix = ConfusionMatrixPartitionMultiSelection(selected_experiments)
confusion_matrix_partition_selection_wg

<br>
Finally, the next cell will open a file selection widget.

Please select the color configuration file (`.yaml`). This file is usually located in the same directory as the experiment logs.

In [ ]:
from dnotebooks.widgets import RegexColorDictFileSelection

colors_file_selection_wg, get_color_dict = RegexColorDictFileSelection()
colors_file_selection_wg

<br><br>
<hr>
<br><br>

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
from dnotebooks.widgets import MultiExperimentMetricsComparisonDashboard

plt.rcParams['figure.figsize'] = [8, 5]

plt.close('all')
plots = MultiExperimentMetricsComparisonDashboard(
    {x.name: [v] for x, v in get_confusion_matrix().items()},
    get_color_dict(),
    title="CIFAR - cat (3)",
    xlabel="Time (s)",
    linestyles=line_styles,
    # external_legend=True
)
display(plots.render())

<br><br>
<hr>

<br><br>